# Predicting used car prices with linear regression

I wanted to understand how linear regression really works, so I wrote it myself in NumPy instead of only calling scikit-learn. I trained it in two ways, Batch Gradient Descent and Stochastic Gradient Descent (SGD). Then I compared both with scikit-learn's `LinearRegression` and `SGDRegressor`.

The data is a cleaned table of used car sales (`vehicle_sales_clean.zip`) with 558,723 rows. Each row is one sale, and I try to predict `SellingPrice`.

My steps: pick the columns, group rare categories, one-hot encode, split into train and test, scale, train the models, and compare the R² scores.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

In [2]:
df=pd.read_csv("vehicle_sales_clean.zip")

In [3]:
df.head()

,Id,Year,Make,Model,Trim,Body,Transmission,VIN,State,ConditionValue,...,PriceFlag,CarAge,PriceDiff,SaleYear,SaleQuarter,PriceDiffPct,PriceStatus,MileageBand,PriceBand,SaleMonth
0,2,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,5.0,...,normal,-1,1000.0,2014,Q4,4.878049,Above MMR,0-25K,20K-30K,2014-12
1,3,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,5.0,...,normal,-1,700.0,2014,Q4,3.365385,Above MMR,0-25K,20K-30K,2014-12
2,4,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,45.0,...,normal,1,-1900.0,2015,Q1,-5.956113,Below MMR,0-25K,30K+,2015-01
3,5,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,41.0,...,normal,0,250.0,2015,Q1,0.909091,At MMR,0-25K,20K-30K,2015-01
4,6,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,43.0,...,normal,0,1000.0,2014,Q4,1.515152,Above MMR,0-25K,30K+,2014-12


In [4]:
print(df.columns.tolist())

['Id', 'Year', 'Make', 'Model', 'Trim', 'Body', 'Transmission', 'VIN', 'State', 'ConditionValue', 'Odometer', 'Color', 'Interior', 'Seller', 'MMR', 'SellingPrice', 'SaleDate', 'ConditionScale', 'PriceFlag', 'CarAge', 'PriceDiff', 'SaleYear', 'SaleQuarter', 'PriceDiffPct', 'PriceStatus', 'MileageBand', 'PriceBand', 'SaleMonth']


## Choosing the columns

I predict `SellingPrice` using these columns: `CarAge`, `Odometer`, `ConditionValue`, `Make`, `Model`, `Body`, `Transmission`, `State`, `Color` and `Interior`.

I left some columns out on purpose:

- `PriceDiff`, `PriceDiffPct`, `PriceStatus`, `PriceFlag` and `PriceBand` are made from the selling price, so the model would see the answer. That is data leakage.
- `Id`, `VIN`, `Seller` and `SaleDate` have too many different values. They would add thousands of columns and tell the model nothing useful about the car.
- `Year`, `SaleYear`, `SaleMonth` and `SaleQuarter` say the same thing as `CarAge`. `MileageBand` says the same thing as `Odometer`.
- I left out `MMR` because it is already a price estimate. With it, the model would mostly copy that estimate.

I also made two new columns, `CarAge2 = (CarAge - mean)²` and `Odometer2 = (Odometer - mean)²`, so the line can bend a little. At first I used the plain square, but `CarAge²` moves almost exactly with `CarAge`, and that made gradient descent very slow. Subtracting the mean first fixed most of that. Now the new column only says how far a car is from the average.

In [5]:
x = df[["CarAge", "Odometer", "ConditionValue",
        "Make", "Model", "Body", "Transmission",
        "State", "Color", "Interior"]]

y = df["SellingPrice"]

x = x.copy()
x["CarAge2"] = (x["CarAge"] - x["CarAge"].mean()) ** 2
x["Odometer2"] = (x["Odometer"] - x["Odometer"].mean()) ** 2

In [6]:
nums=x.select_dtypes(include="number").columns
cats=x.select_dtypes(include="object").columns

In [7]:
print(x[cats].nunique())

Make             67
Model           852
Body             46
Transmission      3
State            38
Color            20
Interior         17
dtype: int64


In [8]:
print(x.isnull().sum())

CarAge            0
Odometer          0
ConditionValue    0
Make              0
Model             0
Body              0
Transmission      0
State             0
Color             0
Interior          0
CarAge2           0
Odometer2         0
dtype: int64


## Grouping rare categories

Some columns have many values that only show up a few times. A category with so few rows gives a weight I can't trust, and it also adds one more column after one-hot encoding.

So `group_rare` keeps the most common values and renames everything else to "other". I kept the top 200 models, the top 50 makes and the top 30 body styles.

The number of unique values changed like this: Make 67 to 51, Model 852 to 201, Body 46 to 31.

In [9]:
def group_rare(col,top_n):
    top=x[col].value_counts().nlargest(top_n).index
    return x[col].where(x[col].isin(top),"other")

x=x.copy()
x["Model"]=group_rare("Model",200)
x["Make"]=group_rare("Make",50)
x["Body"]=group_rare("Body",30)
print(x[cats].nunique())

Make             51
Model           201
Body             31
Transmission      3
State            38
Color            20
Interior         17
dtype: int64


## One-hot encoding

Linear regression only understands numbers, so I turned the text columns into 0/1 columns, one for each category.

I did not use label encoding (1, 2, 3...). Makes and colors have no order, and numbers would tell the model that one make is bigger than another. I also used `drop_first=True` so one category per column is left out and acts as the reference.

After this I have 359 columns.

In [10]:
x=pd.get_dummies(x,columns=cats,drop_first=True,dtype=int)

In [11]:
print(x.shape)

(558723, 359)


In [12]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

## Scaling

I scale every column with `StandardScaler`, including the 0/1 columns.

At first I only scaled the numeric columns and my Batch GD was very slow to learn. The reason is that a rare category column is almost all zeros, so its gradient is tiny and its weight barely moves. After scaling all the columns, they learn at a similar speed.

The scaler learns the mean and the standard deviation from the train set only. Then it uses those same numbers on the test set. If I used the test rows too, the model would get a peek at data it should not have seen.

In [13]:
scaler=StandardScaler()
x_train=x_train.copy()
x_test=x_test.copy()

x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [14]:
x_train=np.array(x_train,dtype=float)
x_test=np.array(x_test,dtype=float)
y_train=np.array(y_train)
y_test=np.array(y_test)

## Batch Gradient Descent

This is my first from-scratch model. For every car it predicts a price like this:

prediction = X * weights + bias

At the start all the weights are zero, so the predictions are bad. Then I repeat the same four steps:

1. Predict the price for all training rows.
2. Measure how wrong I am (the mean squared error).
3. Work out which way each weight should move to make the error smaller.
4. Move each weight a small step in that direction. The size of the step is the learning rate (`lr`).

It is called "batch" because every step uses the whole training set. I print the loss every 500 iterations to check that it keeps going down.

I also added an early stop: if the loss improves by less than 1 between two steps, the loop ends. In my run it never triggered, so it used all 2000 iterations.

My first version scored much lower than sklearn (0.6567). I explain what fixed it in the results at the end.

In [15]:
class linearregression:
    def __init__(self,lr=0.001,iters=1000):
        self.lr=lr
        self.iters=iters
        self.weights=None
        self.bias=None

    def fit(self,x,y):
        n_sample,n_features=x.shape
        self.weights=np.zeros(n_features)
        self.bias=0
        prev_loss = None
        
        for i in range(self.iters):
            error=self.predictl(x)-y
            loss = np.mean(error ** 2)
            if prev_loss is not None and prev_loss - loss < 1:
                print("Stopped at", i, loss)
                break
            prev_loss = loss
            dw=(1/n_sample)*np.dot(x.T,error)
            db=(1/n_sample)*np.sum(error)
            self.weights -= self.lr*dw
            self.bias -= self.lr*db
            if i % 500==0:
                print(i, loss)

    def predictl(self,x):
        return np.dot(x,self.weights)+self.bias

In [16]:
m=linearregression(lr=0.3,iters=2000)
m.fit(x_train,y_train)
pred=m.predictl(x_test)
print(r2_score(y_test,pred))

0 280689901.59317464
500 20099858.01742411
1000 20096786.74578578
1500 20094620.602556545
0.7906339328973656


In [17]:
model=LinearRegression()
model.fit(x_train,y_train)
m_pred=model.predict(x_test)
print(r2_score(y_test,m_pred))

0.7906518923362325


## Stochastic Gradient Descent

This is almost the same as Batch GD, with one change. Instead of using all the rows for one step, I update the weights after every single row.

One pass over all the training rows is called an epoch. I shuffle the rows at the start of each epoch so the updates don't come in the same order every time.

Each update is noisy, but there are a lot of them, so I only need a few epochs. The downside is that my version loops over the rows in plain Python, so it is slow on this much data.

One more thing I learned: SGD is very sensitive to the learning rate. I explain what went wrong in the results at the end.

In [18]:
class sgd:
    def __init__(self,lr=0.01,iters=50):
        self.lr=lr
        self.iters=iters
        self.weights=None
        self.bias=None

    def fit(self,x,y):
        n_sample,n_features=x.shape
        self.weights=np.zeros(n_features)
        self.bias=0
        for _ in range(self.iters):
            indcies=np.random.permutation(n_sample)
            for i in indcies:
                xi=x[i]
                yi=y[i]
                error=self.predicts(xi)-yi
                dw=xi*error
                db=error
                self.weights -= self.lr*dw
                self.bias -= self.lr*db

    def predicts(self,x):
        return np.dot(x,self.weights)+self.bias

In [19]:
my_m=sgd(lr=0.00001, iters=5)
my_m.fit(x_train,y_train)
my_pred=my_m.predicts(x_test)
print(r2_score(y_test,my_pred))

0.7900375765526622


In [20]:
s_m=SGDRegressor(eta0=0.00001,max_iter=5,learning_rate="constant",random_state=42)
s_m.fit(x_train,y_train)
s_pred=s_m.predict(x_test)
print(r2_score(y_test,s_pred))

0.7886187211755018


c:\Users\aliad\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1612: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


## Results

R² on the test set:

| Model | Training method | R² |
|---|---|---|
| My `linearregression` (lr=0.3, 2000 iterations) | Batch GD | 0.7906 |
| sklearn `LinearRegression` | Closed form | 0.7907 |
| My `sgd` (lr=0.00001, 5 epochs) | SGD | 0.7900 |
| sklearn `SGDRegressor` (eta0=0.00001, constant learning rate, 5 epochs) | SGD | 0.7886 |

sklearn's `LinearRegression` solves the problem directly, so I treat its score as the best I can get with this model. My Batch GD ends very close to it (0.7906), and its loss was almost flat at about 20.1 million by the end. The early stop never triggered, so it used all 2000 iterations.

My first Batch GD result was much lower, 0.6567. That run only scaled the numeric columns, used the plain squared features, and had lr=0.1 with 5000 iterations. Then I scaled all the columns and centered the squared features. With lr=0.3 and 2000 iterations it reached 0.7906. sklearn's score did not change (0.7907), which makes sense because it gets the same information either way.

SGD was harder. After I scaled all the columns, the rare one-hot columns got large values for the few rows that have them. SGD updates after every single row, so when one of those rows came with a big learning rate, it made a huge step and the model exploded. This happened to my SGD and to sklearn's.

Lowering the learning rate fixed my SGD. For sklearn I also had to set the learning rate to constant. By default it keeps shrinking the learning rate, so with a small starting value it barely learned. Now my SGD gets 0.7900 and sklearn's gets 0.7886, both close to the 0.7907 from `LinearRegression`. They needed only 5 epochs, while Batch GD needed about 2000 iterations. sklearn still gives a ConvergenceWarning because 5 epochs is a very small number.

### Earlier experiments (not shown in this notebook)

| Experiment | R² |
|---|---|
| Basic features | 0.66 |
| After adding squared features | 0.68 |
| After keeping more categories | 0.79 |
| Log-transformed target | 0.64 |
| Tree-based model (`HistGradientBoostingRegressor`) | 0.93 |

The log transform made things a bit worse (0.64 vs 0.66). Squared features and more categories helped.

### What linear regression can't do

It draws a straight-line relationship, so it misses curves and interactions. For example, age affects a luxury car differently than an economy car.

### Next

A quick tree-based model test reached 0.93, so I want to look at tree models when I study them.